# Mamografias em Minas Gerais — Análise Exploratória de Dados

**Projeto:** TCC — Análise de Dados de Câncer de Mama em Municípios de Minas Gerais  
**Metodologia:** KDD (Knowledge Discovery in Databases)  
**Etapa:** Análise Exploratória de Dados (EDA)  
**Escopo:** Volume de mamografias por ano e faixa etária (2013–2025) — municípios e regionais de saúde de MG

---
## 1. Imports

In [ ]:
import json
import unicodedata
import warnings
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

# Seleciona o renderer adequado automaticamente
for renderer in ("plotly_mimetype", "vscode", "notebook_connected", "notebook"):
    if renderer in pio.renderers:
        pio.renderers.default = renderer
        break

---
## 2. Configurações Gerais

Centraliza todos os parâmetros configuráveis: caminhos, filtros, paletas de cores e constantes de layout.  
Ajuste aqui qualquer parâmetro antes de executar o notebook.

In [ ]:
# ── Caminhos dos arquivos de entrada ─────────────────────────────────────────
CSV_MAMOGRAFIA         = Path("/home/anna/tcc/Tratamento/Mamografia/tabela_mamografia.csv")
CSV_MUNICIPIOS_REGIONAIS = Path("/home/anna/tcc/Tratamento/Municípios/tabela_municipios_regionais.csv")

# ── Período e categorias ─────────────────────────────────────────────────────
ANOS = list(range(2013, 2026))

FAIXAS_ETARIAS = ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70+"]

TOP_N = 10   # número de municípios/regionais no ranking

# ── Cores dos gráficos ───────────────────────────────────────────────────────
COR_BARRAS_MUNICIPIOS = "#2D9CDB"
COR_BARRAS_REGIONAIS  = "#D7263D"

CORES_TENDENCIA = [
    "#002F3E", "#5bc9e7", "#0c70d5", "#4a7189",
    "#DBB347", "#E79417", "#D01C08", "#9B3E29",
]

# ── Layout dos gráficos ───────────────────────────────────────────────────────
LARGURA_GRAFICO       = 1200
ALTURA_TENDENCIA      = 680
ALTURA_RANKING        = 600
FONTE_TITULO          = 22
FONTE_EIXO_TITULO     = 16
FONTE_EIXO_TICK       = 13
FONTE_TEXTO_BARRA     = 13

# ── Escala quebrada do gráfico de tendência ──────────────────────────────────
# Valores abaixo de LIMITE_QUEBRA são exibidos linearmente;
# acima, são compactados pelo FATOR_COMPACTACAO para melhor leitura.
LIMITE_QUEBRA    = 1000
FATOR_COMPACTACAO = 0.08
PASSO_TENDENCIA   = 25000

---
## 3. Funções Auxiliares

Funções utilitárias reutilizadas ao longo de todo o notebook. Definidas uma única vez aqui.

In [ ]:
def _norm_txt(valor: str) -> str:
    """Remove acentos e converte para minúsculas (para comparações)."""
    texto = unicodedata.normalize("NFKD", str(valor).strip().lower())
    return " ".join(c for c in texto if not unicodedata.combining(c)).split().__repr__()[2:-2] if False else            " ".join("".join(c for c in texto if not unicodedata.combining(c)).split())


def _find_col(df: pd.DataFrame, candidatos: list):
    """Retorna a primeira coluna do df cujo nome normalizado bata com algum candidato."""
    mapa = {_norm_txt(col): col for col in df.columns}
    for c in candidatos:
        found = mapa.get(_norm_txt(c))
        if found:
            return found
    return None


def _norm_cod7(serie: pd.Series) -> pd.Series:
    """Normaliza código IBGE para 7 dígitos com zero à esquerda."""
    return (
        pd.to_numeric(serie, errors="coerce")
        .round().astype("Int64").astype(str)
        .str.replace("<NA>", "", regex=False)
        .str.zfill(7)
    )


def _to_numeric_cols(df: pd.DataFrame, colunas: list) -> pd.DataFrame:
    """Converte colunas para numérico in-place (fillna 0)."""
    df = df.copy()
    existentes = [c for c in colunas if c in df.columns]
    if existentes:
        df[existentes] = df[existentes].apply(pd.to_numeric, errors="coerce").fillna(0)
    return df


def _fmt_int(valor) -> str:
    """Formata inteiro com separador de milhar brasileiro (ponto)."""
    try:
        return f"{int(valor):,}".replace(",", ".")
    except Exception:
        return str(valor)


def _fmt_pct(valor) -> str:
    """Formata percentual com vírgula decimal."""
    try:
        return f"{float(valor):.2f}%".replace(".", ",")
    except Exception:
        return str(valor)


def _escala_quebrada(valor: float) -> float:
    """Compacta valores acima de LIMITE_QUEBRA para o gráfico de tendência."""
    valor = float(valor)
    if valor <= LIMITE_QUEBRA:
        return valor
    return LIMITE_QUEBRA + (valor - LIMITE_QUEBRA) * FATOR_COMPACTACAO


def montar_base_longa(df: pd.DataFrame) -> pd.DataFrame:
    """Transforma colunas YYYY_faixa em formato longo (ano, faixa_etaria, exames)."""
    base   = df[["Municipio", "Codigo", "Regional"]].copy()
    partes = []
    for ano in ANOS:
        for faixa in FAIXAS_ETARIAS:
            col = f"{ano}_{faixa}"
            if col not in df.columns:
                continue
            partes.append(base.assign(
                ano=ano,
                faixa_etaria=faixa,
                exames=pd.to_numeric(df[col], errors="coerce").fillna(0),
            ))
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()


print("✅ Funções auxiliares definidas.")

---
## 4. Carregamento e Preparação dos Dados

Leitura do CSV de mamografias, integração com a tabela de regionais de saúde e preparação das séries de análise.

In [ ]:
# ── Validação de arquivos ─────────────────────────────────────────────────────
for path in (CSV_MAMOGRAFIA, CSV_MUNICIPIOS_REGIONAIS):
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")

# ── Leitura e padronização de colunas ────────────────────────────────────────
df = pd.read_csv(CSV_MAMOGRAFIA)

col_mun = _find_col(df, ["Municípios", "MUNICÍPIOS", "Municipios", "Município", "MUNICIPIO"])
col_cod = _find_col(df, ["Código", "CÓDIGO", "Codigo", "CODIGO"])

if col_mun is None or col_cod is None:
    raise KeyError(f"Colunas de município/código não encontradas. Colunas disponíveis: {df.columns.tolist()}")

df = df.rename(columns={col_mun: "Municipio", col_cod: "Codigo"})
df["Municipio"] = df["Municipio"].astype(str).str.strip()
df["Codigo"]    = _norm_cod7(df["Codigo"])

# ── Integração com regionais ─────────────────────────────────────────────────
ref_reg = (
    pd.read_csv(CSV_MUNICIPIOS_REGIONAIS, dtype={"codigo_ibge": "string"})
    [["codigo_ibge", "regional_original"]]
    .drop_duplicates()
    .rename(columns={"regional_original": "Regional"})
)
ref_reg["codigo_ibge"] = _norm_cod7(ref_reg["codigo_ibge"])

df = df.merge(ref_reg, left_on="Codigo", right_on="codigo_ibge", how="left")
df["Regional"] = df["Regional"].fillna("Sem regional")
df = df.drop(columns=["codigo_ibge"], errors="ignore")

# ── Conversão de colunas numéricas ───────────────────────────────────────────
cols_total_ano  = [f"total_{ano}" for ano in ANOS if f"total_{ano}" in df.columns]
cols_faixa_ano  = [f"{ano}_{faixa}" for ano in ANOS for faixa in FAIXAS_ETARIAS if f"{ano}_{faixa}" in df.columns]
cols_numericas  = cols_total_ano + cols_faixa_ano + (["total"] if "total" in df.columns else [])
df = _to_numeric_cols(df, cols_numericas)

if not cols_total_ano:
    raise KeyError("Nenhuma coluna total_YYYY encontrada. Verifique o CSV de mamografia.")

df["total_calculado_anos"] = df[cols_total_ano].sum(axis=1)
df["diferenca_total"]      = (df["total"] - df["total_calculado_anos"]) if "total" in df.columns else 0

# ── Base longa (ano × faixa_etaria) ─────────────────────────────────────────
df_long = montar_base_longa(df)

# ── Resumos por dimensão ─────────────────────────────────────────────────────
resumo_por_faixa = (
    df_long.groupby("faixa_etaria", as_index=False)["exames"]
    .sum().rename(columns={"faixa_etaria": "Faixa etária", "exames": "Exames"})
    .set_index("Faixa etária").reindex(FAIXAS_ETARIAS).reset_index()
    .fillna({"Exames": 0})
)

col_total = "total" if "total" in df.columns else "total_calculado_anos"

resumo_por_regional = (
    df.groupby("Regional", as_index=False)[col_total]
    .sum().rename(columns={col_total: "Exames"})
    .sort_values("Exames", ascending=False).reset_index(drop=True)
)

resumo_por_municipio = (
    df[["Municipio", "Codigo", "Regional", col_total, "total_calculado_anos", "diferenca_total"]]
    .rename(columns={
        "Municipio": "Município", "Codigo": "Código",
        col_total: "Exames",
        "total_calculado_anos": "Total calculado (anos)",
        "diferenca_total": "Diferença",
    })
    .sort_values("Exames", ascending=False).reset_index(drop=True)
)

top_municipios = resumo_por_municipio.head(TOP_N).copy()
top_regionais  = resumo_por_regional.head(TOP_N).copy()

total_geral     = int(df[col_total].sum())
total_calculado = int(df["total_calculado_anos"].sum())
total_diferenca = int(df["diferenca_total"].abs().sum())

print("╔══════════════════════════════════════════════════╗")
print("║           Resumo do carregamento                ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Municípios carregados:        {len(df):<18}║")
print(f"║  Total geral de mamografias:   {_fmt_int(total_geral):<18}║")
print(f"║  Total calculado (soma anual): {_fmt_int(total_calculado):<18}║")
print(f"║  Diferença abs. (total vs ano):{_fmt_int(total_diferenca):<18}║")
print(f"║  Com regional identificada:    {int((df['Regional'] != 'Sem regional').sum()):<18}║")
print("╚══════════════════════════════════════════════════╝")

---
## 5. Análise Inicial dos Dados

Inspeção rápida da estrutura do DataFrame para verificar colunas, tipos e valores ausentes.

In [ ]:
display(Markdown(f"**Shape:** {df.shape[0]} linhas × {df.shape[1]} colunas"))

display(Markdown("**Primeiras linhas:**"))
display(df[["Municipio", "Codigo", "Regional"] + cols_total_ano[:5]].head(3))

missing = df.isnull().sum()
missing = missing[missing > 0]
if not missing.empty:
    display(Markdown("**Valores ausentes por coluna:**"))
    display(missing.to_frame("Nulos").style.background_gradient(cmap="Reds"))
else:
    display(Markdown("✅ **Sem valores ausentes nas colunas principais.**"))

---
## 6. Tabelas Estatísticas

### 6.1 Distribuição por faixa etária

In [ ]:
total_faixas = int(resumo_por_faixa["Exames"].sum())
df_faixa_pct = resumo_por_faixa.copy()
df_faixa_pct["Percentual (%)"] = df_faixa_pct["Exames"] / total_faixas * 100

linha_total_faixas = pd.DataFrame([{
    "Faixa etária": "Total",
    "Exames": total_faixas,
    "Percentual (%)": 100.0,
}])
df_faixa_display = pd.concat([df_faixa_pct, linha_total_faixas], ignore_index=True)

display(Markdown("**Exames por faixa etária e percentual de participação:**"))
display(
    df_faixa_display.style
    .format({"Exames": _fmt_int, "Percentual (%)": _fmt_pct})
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
    .hide(axis="index")
)

### 6.2 Estatísticas por ano

In [ ]:
# Pivô ano × faixa para calcular estatísticas anuais
tendencia_pivot = (
    df_long.groupby(["ano", "faixa_etaria"], as_index=False)["exames"]
    .sum()
    .pivot(index="ano", columns="faixa_etaria", values="exames")
    .fillna(0)
    .reindex(columns=FAIXAS_ETARIAS)
    .fillna(0)
)

df_estatisticas_ano = pd.DataFrame({
    "Ano"                            : tendencia_pivot.index.astype(str),
    "Exames (soma das faixas)"       : tendencia_pivot.sum(axis=1).astype(int).values,
    "Média de exames (entre faixas)" : tendencia_pivot.mean(axis=1).round(0).astype(int).values,
    "Desvio padrão (entre faixas)"   : tendencia_pivot.std(axis=1).round(0).astype(int).values,
})

linha_total_ano = pd.DataFrame([{
    "Ano"                            : "Total",
    "Exames (soma das faixas)"       : int(df_estatisticas_ano["Exames (soma das faixas)"].sum()),
    "Média de exames (entre faixas)" : int(df_estatisticas_ano["Média de exames (entre faixas)"].mean()),
    "Desvio padrão (entre faixas)"   : int(df_estatisticas_ano["Desvio padrão (entre faixas)"].mean()),
}])
df_est_ano_display = pd.concat([df_estatisticas_ano, linha_total_ano], ignore_index=True)

display(Markdown("**Estatísticas por ano (soma das faixas, média e desvio padrão):**"))
display(
    df_est_ano_display.style
    .format({
        "Exames (soma das faixas)"       : _fmt_int,
        "Média de exames (entre faixas)" : _fmt_int,
        "Desvio padrão (entre faixas)"   : _fmt_int,
    })
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
    .hide(axis="index")
)

---
## 7. Visualizações

### 7.1 Tendência anual por faixa etária

Gráfico de linhas com escala quebrada: valores até {LIMITE_QUEBRA} exibidos linearmente; acima, compactados pelo fator {FATOR_COMPACTACAO} para facilitar a leitura de faixas com volumes muito distintos.

In [ ]:
maior_valor  = int(tendencia_pivot.max().max()) if not tendencia_pivot.empty else 0
teto_tend    = int(((maior_valor + PASSO_TENDENCIA - 1) // PASSO_TENDENCIA) * PASSO_TENDENCIA)
ticks_reais  = sorted({0, LIMITE_QUEBRA} | set(range(PASSO_TENDENCIA, teto_tend + PASSO_TENDENCIA, PASSO_TENDENCIA)))
ticks_plot   = [_escala_quebrada(v) for v in ticks_reais]

fig_tend = go.Figure()
for idx, faixa in enumerate(FAIXAS_ETARIAS):
    if faixa not in tendencia_pivot.columns:
        continue
    serie = tendencia_pivot[faixa]
    cor   = CORES_TENDENCIA[idx % len(CORES_TENDENCIA)]
    fig_tend.add_trace(go.Scatter(
        x=tendencia_pivot.index.astype(str),
        y=[_escala_quebrada(v) for v in serie],
        mode="lines+markers",
        name=faixa,
        line=dict(color=cor, width=2.7),
        marker=dict(color=cor, size=9),
        hovertext=[f"{faixa}: {_fmt_int(v)}" for v in serie],
        hovertemplate="%{x}<br>%{hovertext}<extra></extra>",
        cliponaxis=False,
    ))

fig_tend.update_layout(
    title=dict(
        text="Tendência anual por faixa etária",
        font=dict(size=FONTE_TITULO), x=0.5,
    ),
    xaxis=dict(
        title="Ano",
        title_font=dict(size=FONTE_EIXO_TITULO),
        tickfont=dict(size=FONTE_EIXO_TICK),
        tickangle=0,
    ),
    yaxis=dict(
        title="Quantidade de exames",
        title_font=dict(size=FONTE_EIXO_TITULO),
        tickfont=dict(size=FONTE_EIXO_TICK),
        tickmode="array",
        tickvals=ticks_plot,
        ticktext=[_fmt_int(v) for v in ticks_reais],
        gridcolor="#E5ECF6",
        range=[0, _escala_quebrada(teto_tend)],
    ),
    template="plotly_white",
    height=ALTURA_TENDENCIA,
    width=LARGURA_GRAFICO,
    margin=dict(l=80, r=40, t=80, b=55),
    legend_title_text="Faixa etária",
    hovermode="x",
)
fig_tend.show()

### 7.2 Ranking de municípios por volume de exames

In [ ]:
top_mun_sorted = top_municipios.sort_values("Exames")

fig_mun = go.Figure(go.Bar(
    x=top_mun_sorted["Exames"],
    y=top_mun_sorted["Município"],
    orientation="h",
    marker_color=COR_BARRAS_MUNICIPIOS,
    text=[_fmt_int(v) for v in top_mun_sorted["Exames"]],
    textposition="outside",
    textfont=dict(size=FONTE_TEXTO_BARRA),
    cliponaxis=False,
    constraintext="none",
))
fig_mun.update_layout(
    title=dict(
        text=f"Top {TOP_N} municípios por número de exames",
        font=dict(size=FONTE_TITULO), x=0.5,
    ),
    xaxis=dict(
        title="Quantidade de exames",
        title_font=dict(size=FONTE_EIXO_TITULO),
        tickfont=dict(size=FONTE_EIXO_TICK),
        range=[0, int(top_mun_sorted["Exames"].max() * 1.20)],
        showgrid=True, gridcolor="#E5ECF6",
    ),
    yaxis=dict(
        title="Município",
        title_font=dict(size=FONTE_EIXO_TITULO),
        tickfont=dict(size=FONTE_EIXO_TICK),
        automargin=True,
    ),
    template="plotly_white",
    height=ALTURA_RANKING,
    width=LARGURA_GRAFICO,
    margin=dict(l=20, r=120, t=70, b=20),
)
fig_mun.show()

### 7.3 Ranking de regionais por volume de exames

In [ ]:
top_reg_sorted = top_regionais.sort_values("Exames")

fig_reg = go.Figure(go.Bar(
    x=top_reg_sorted["Exames"],
    y=top_reg_sorted["Regional"],
    orientation="h",
    marker_color=COR_BARRAS_REGIONAIS,
    text=[_fmt_int(v) for v in top_reg_sorted["Exames"]],
    textposition="outside",
    textfont=dict(size=FONTE_TEXTO_BARRA),
    cliponaxis=False,
    constraintext="none",
))
fig_reg.update_layout(
    title=dict(
        text=f"Top {TOP_N} regionais por número de exames",
        font=dict(size=FONTE_TITULO), x=0.5,
    ),
    xaxis=dict(
        title="Quantidade de exames",
        title_font=dict(size=FONTE_EIXO_TITULO),
        tickfont=dict(size=FONTE_EIXO_TICK),
        range=[0, int(top_reg_sorted["Exames"].max() * 1.20)],
        showgrid=True, gridcolor="#E5ECF6",
    ),
    yaxis=dict(
        title="Regional de Saúde",
        title_font=dict(size=FONTE_EIXO_TITULO),
        tickfont=dict(size=FONTE_EIXO_TICK),
        automargin=True,
    ),
    template="plotly_white",
    height=ALTURA_RANKING,
    width=LARGURA_GRAFICO,
    margin=dict(l=20, r=120, t=70, b=20),
)
fig_reg.show()

---
## 8. Conclusões da EDA — Mamografias

### Tendência temporal por faixa etária
- As faixas **40–49** e **50–59** concentram o maior volume absoluto de exames, alinhado às diretrizes do SUS para rastreamento mamográfico.
- Faixas mais jovens (0–29) apresentam volumes muito baixos, condizente com a baixa indicação clínica nessas idades.
- A escala quebrada do gráfico de tendência evidencia a disparidade entre as faixas de maior e menor volume.

### Ranking de municípios e regionais
- Os maiores volumes estão concentrados nos municípios da Região Metropolitana de Belo Horizonte e municípios-sede de regionais de saúde.
- A distribuição por regional reflete tanto a densidade populacional quanto a oferta de serviços de saúde na região.

### Qualidade dos dados
- A diferença entre a coluna `total` e a soma das colunas anuais é monitorada (`diferenca_total`) para detectar inconsistências de registro.

> **Próxima etapa (PFC II):** Cruzamento do volume de mamografias com indicadores de mortalidade e incidência para análise bivariada e clustering K-means.